#Bronze Layer Script ###

In [0]:
%python
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.types import FloatType, IntegerType
from pyspark.sql import functions as F
from pyspark.sql.functions import lower, regexp_replace, split, array_join
from pyspark.sql.functions import col, substring_index
from pyspark.sql.functions import when, col
from pyspark.sql.functions import col, when, round as spark_round


##Data Reading


In [0]:
%python

input_path="/Volumes/ai_de_assignment/bronze/ai_de_assignment_vol/amazon.csv"
# handle escape,quote and multiline
df = (spark.read
      .option("header", True)
      .option("inferSchema", True)
      .option("multiLine", True)
      .option("escape", '"')
      .option("quote", '"')
      .csv(input_path)
     )


In [0]:
%python
display(df.limit(10))


###Row count

In [0]:
%python
df.count()


### Feature description (data type check)


In [0]:
%python
data = [
    ("product_id", "object", "Unique identifier for each product"),
    ("product_name", "object", "Name of the product"),
    ("category", "object", "Category to which the product belongs"),
    ("discounted_price", "object", "Discounted price of the product"),
    ("actual_price", "object", "Original price of the product before discounts"),
    ("discount_percentage", "object", "Percentage of the discount provided on the product"),
    ("rating", "object", "Average rating given to the product by users"),
    ("rating_count", "object", "Number of users who have rated the product"),
    ("about_product", "object", "Description or details about the product"),
    ("user_id", "object", "Unique identifier for the user who wrote the review"),
    ("user_name", "object", "Name of the user who wrote the review"),
    ("review_id", "object", "Unique identifier for each user review"),
    ("review_title", "object", "Short title or summary of the user review"),
    ("review_content", "object", "Full content of the user review"),
    ("img_link", "object", "URL link to the product's image"),
    ("product_link", "object", "URL link to the product's page on Amazon's official website")
]

columns = ["Feature Name", "Data Type", "Description"]

descriptive_df = spark.createDataFrame(data, columns)

display(descriptive_df)

#Silver Layer Script

##Data Cleaning 

### Check for null entries

In [0]:
%python
null_count_df = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_count_df) 

### Result : Two records have null values in the Rating Count column

###Convert the data to the correct format to enable accurate calculations and exclude invalid characters.

In [0]:
%python
# Remove unwanted characters and try casting safely
df = df.withColumn(
    'discounted_price',
    F.expr("try_cast(regexp_replace(discounted_price, '₹|,', '') AS FLOAT)")
)

df = df.withColumn(
    'actual_price',
    F.expr("try_cast(regexp_replace(actual_price, '₹|,', '') AS FLOAT)")
)

df = df.withColumn(
    'discount_percentage',
    F.expr("try_cast(regexp_replace(discount_percentage, '%', '') AS FLOAT)")
)

df = df.withColumn(
    'rating',
    F.expr("try_cast(regexp_replace(rating, '\\|', '') AS FLOAT)")
)

df = df.withColumn(
    'rating_count',
    F.expr("try_cast(regexp_replace(rating_count, ',', '') AS INT)")
)


###Note : During conversion, the rating column was found to contain the | character, which has been handled.

###Clean and preprocess the text. This steps involve:

1. Lowercase.
2. Removing punctuation and special characters.
3. Handle html tags review content.


In [0]:
%python
df = df.withColumn('product_name', 
                   regexp_replace(lower(col('product_name')), r'[^a-zA-Z0-9\s]', ''))

df = df.withColumn('about_product', 
                   regexp_replace(lower(col('about_product')), r'[^a-zA-Z0-9\s]', ''))


df = df.withColumn(
    'review_content',
    regexp_replace(  # Step 3: special characters
        regexp_replace(  # Step 2: remove HTML tags
            lower(col('review_content')),  # Step 1: convert to lowercase
            r'<[^>]+>', ''
        ),
        r'[^a-zA-Z0-9\s]', ''
    )
)

df = df.withColumn('category', lower(col('category')))
df = df.withColumn('user_name', lower(col('user_name')))
df = df.withColumn('review_title', lower(col('review_title')))
df = df.withColumn('product_link', lower(col('product_link')))



In [0]:
%python
#product link short
df = df.withColumn('product_link', substring_index(col('product_link'), '/ref=', 1))


###Drop Duplicate

In [0]:
%python
df = df.dropDuplicates()

In [0]:
%python
df.count()

#Data transformation layer

### Sentiment column : 
If we need to build logic based on a combination of rating count and rating, then the logic below needs to be changed.

In [0]:
%python

df = df.withColumn("rating_rounded", spark_round(col("rating"), 2))

#  Sentiment logic
df = df.withColumn(
    "sentiment",when(col("rating_rounded") < 3, "negative")
    .when((col("rating_rounded") >= 3) & (col("rating_rounded") < 4), "neutral")
    .when(col("rating_rounded") >= 4, "positive")
)

# 4. Drop temporary column
df = df.drop("rating_rounded")


# KPI calculation 
df = df.withColumn("discount_amount", col("actual_price") - col("discounted_price")) \
        .withColumn("revenue_potential", col("discounted_price") * col("rating_count"))




In [0]:
%python


# desired column order
columns_order = [
    "product_id", "product_name", "category", "discounted_price", "actual_price",
    "discount_percentage","discount_amount","revenue_potential", "rating", "rating_count", "sentiment" ,"about_product",
    "user_id", "user_name", "review_id", "review_title", "review_content",
    "img_link", "product_link" 
]

df = df.select([col(c) for c in columns_order])


## Data write in Sliver Layer

In [0]:
%python
table_name = "ai_de_assignment.silver.silver_amazon_reviews"

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)


In [0]:
%python
spark.sql(f"SELECT * FROM {table_name} LIMIT 5").display(truncate=False)


In [0]:
%python
output_path="/Volumes/ai_de_assignment/silver/ai_de_assignment_cleaned/sliver_amazon_review"


df.coalesce(1).write \
  .option("header", True) \
  .option("quote", '"') \
  .option("escape", '"') \
  .option("multiLine", True) \
  .mode("overwrite") \
  .csv(output_path)


#Duplicate Observation 

##For calculating the KPI, we need to be careful about duplication caused by multiple reasons:

1. Some product IDs have multiple rows for the actual price. These should either be summed or properly aggregated before calculating the selling price.
2. Some product IDs have multiple rows for the rating count. These should either be summed or the maximum value should be taken per product.
3. In some cases, the review ID and user ID are swapped. We need to identify distinct user IDs or review IDs per product.
4. Some products have more than one image link, and review content may contain image or product links, which causes duplicate records.

#Spark Streaming -->Demo


In [0]:
%skip
input_path = "/Volumes/ai_de_assignment/bronze/ai_de_assignment_vol"

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("cloudFiles.schemaLocation",
                "/Volumes/ai_de_assignment/bronze/ai_de_assignment_vol/_schemas/amazon/")
        .load(input_path)
)


In [0]:
%skip

output = "ai_de_assignment.silver"
checkpoint_path = "/Volumes/ai_de_assignment/silver/ai_de_assignment_cleaned/_checkpoints/amazon"

query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", checkpoint_path)
         .outputMode("append")
         .trigger(once=True)  
         .table("ai_de_assignment.silver.silver_amazon_reviews_delta")  
        )

